Notes:

*   Steps 1 through 3 must be run when resuming from a checkpoint.
*   To use checkpoints, download the csv file from the processing section in the google drive and upload it to the files tab on the left of this google colab workspace.


1. Libraries

In [ ]:
from datetime import datetime, date, timedelta
from tqdm import tqdm
from google.colab import drive

import numpy as np
import pandas as pd

2. Load Data

In [ ]:
"""
To make these data import code lines work:

1. Go to "Honors Thesis" Shared Drive.
2. Click the three vertical dots to the right of the "Extracted Data Organized" folder.
3. Click "Organize."
4. Click "Add shortcut."
5. Click the "Add" button to the right of "My Drive."

Thesis steps create a shortcut in My Drive which points to the folder of data.
"""

drive.mount('/content/drive')

adim = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/arena_dim.csv")
afact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/arena_fact.csv")
dfact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/daily_fact.csv")
gfact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/game_fact.csv")
ifact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/IL_update_fact.csv")
pdim = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/player_dim.csv")
tdim = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/team_dim.csv")

Mounted at /content/drive


/tmp/ipython-input-495893950.py:18: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  gfact = pd.read_csv("/content/drive/My Drive/Extracted Data Organized/game_fact.csv")


3. Define Important Functions

In [ ]:
def string_to_date(date_str): #convert strings to datetime
    return datetime.strptime(date_str, "%Y-%m-%d")

def date_to_string(date): #convert datetime to strings
    return date.strftime('%Y-%m-%d')

"""
rs_starts and rs_ends list the start and end of the regular seasons from 2017-18
to 2024-25. Each index from both lists is a single season.
"""
rs_starts = [
    datetime(2017, 10, 17),
    datetime(2018, 10, 16),
    datetime(2019, 10, 22),
    datetime(2020, 12, 22),
    datetime(2021, 10, 19),
    datetime(2022, 10, 18),
    datetime(2023, 10, 24),
    datetime(2024, 10, 22)
]

rs_ends = [
    datetime(2018, 4, 11),
    datetime(2019, 4, 10),
    datetime(2020, 4, 15),
    datetime(2021, 4, 16),
    datetime(2022, 4, 10),
    datetime(2023, 4, 9),
    datetime(2024, 4, 14),
    datetime(2025, 4, 13)
]
"""
For a datetime object passed, the function returns True if the datetime is within
all relevant regular season (season start and end dates inclusive) and False otherwise.
"""
def check_date_in_rs(datetime):
    for i in range(len(rs_starts)):
        if datetime >= rs_starts[i] and datetime <= rs_ends[i]:
            return True
    return False

**Preprocessing**

4. Create a table of all significant days for all players from daily_fact.csv. This will be the foundation for the preprocessed records.

In [ ]:
#flag record where the player is recovering from an injury (not relevant)
dfact.loc[:, 'date'] = dfact.loc[:, 'date'].apply(string_to_date)

player_keys = dfact['player_key']
dates = dfact['date']
status = dfact['injured?']

recover_flag = []

for pk in tqdm(dfact['player_key'].unique()):
  player_dates = []
  player_status = []

  for i in range(len(player_keys)):
    if player_keys[i] == pk:
      player_dates.append(dates[i])
      player_status.append(status[i])

  for i in range(len(player_dates)):
    yesterday = player_dates[i] - timedelta(days=1)
    if player_dates[i-1] == yesterday:
      yesterday_status = player_status[i-1]
      if yesterday_status and player_status[i]:
        recover_flag.append('Y')
      else:
        recover_flag.append('N')
    else:
      recover_flag.append('E')

print('items added to recover_flag: ', len(recover_flag))
print('rows in dfact: ',dfact.shape[0])

recovery_col = pd.DataFrame({'recovering?':recover_flag})
dfact = pd.concat([dfact, recovery_col], axis=1, ignore_index=True)
dfact.head()


100%|██████████| 1199/1199 [1:03:33<00:00,  3.18s/it]


items added to recover_flag:  1519374
rows in dfact:  1519374


,0,1,2,3,4,5,6
0,1713,23,2017-09-17 00:00:00,14844,7019,True,E
1,1713,23,2017-09-18 00:00:00,14845,7020,True,Y
2,1713,23,2017-09-19 00:00:00,14846,7021,True,Y
3,1713,23,2017-09-20 00:00:00,14847,7022,True,Y
4,1713,23,2017-09-21 00:00:00,14848,7023,True,Y


In [ ]:
dfact = dfact.rename(columns={0:'player_key', 1:'team_key', 2:'date', 3:'age', 4:'experience', 5:'injured?', 6:'recoverying?'})
dfact.to_csv('p1cp1.csv', index=False) #preprocessing1 checkpoint 1 saved as p1cp1.csv and available in the Preprocessing folder

Checkpoint 1

In [ ]:
dfact = pd.read_csv('/content/p1cp1.csv')
dfact['recoverying?'].value_counts()

# Y: recovering (on that day, that player was injured and was injured the day before)
# N: not recovering (was either healthy on that day, healthy the day before, or healthy both days)
# E: there was no day before to reference

,count
recoverying?,
N,1266698
Y,251477
E,1199


In [ ]:
# new DataFrame from gfact where there are only games from the regular seasons
def get_season_df(i):
  temp = gfact.copy()
  temp['game_date'] = temp['game_date'].apply(string_to_date)
  temp = temp.loc[temp['game_date'] >= rs_starts[i], :]
  temp = temp.loc[temp['game_date'] <= rs_ends[i]]
  return temp

rsfact = pd.DataFrame(columns=gfact.columns)

for i in range(8):
  temp = get_season_df(i)
  rsfact = pd.concat([rsfact, temp], axis=0, ignore_index=True)


rsfact['game_date'] = rsfact['game_date'].apply(date_to_string)
print('Shape of original gfact', gfact.shape)
print('Shape of new rsfact', rsfact.shape)

/tmp/ipython-input-3780612761.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rsfact = pd.concat([rsfact, temp], axis=0, ignore_index=True)


Shape of original gfact (262472, 43)
Shape of new rsfact (234649, 43)


In [ ]:
"""
Create a dictionary of dictionaries for relevant windows of time for players across seasons.
Hierarchy:
1. Player
  2. Season
    3. Start of Season and End of Season
"""

seasons = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] #the year that each season ended

all_injuries = ifact.loc[ifact['relinquished/acquired'] == 'r', ['player_key', 'date']]

player_windows = {}

for pk in tqdm(dfact['player_key'].unique()):
  player_injuries = all_injuries.loc[all_injuries['player_key'] == pk] # all this player's injuries
  player_injuries.loc[:, 'date'] = player_injuries.loc[:, 'date'].apply(string_to_date) # convert dates from string to datetime
  game_dates = rsfact.loc[rsfact['player_key'] == pk, ['game_date', 'season']] # all game dates in all season for thie player
  player = {} #dictionary, keys are seasons, items are lists of start and end dates of each season

  for i in range(8):
    season_games = game_dates.loc[game_dates['season'] == seasons[i], 'game_date']
    if season_games.shape[0] == 0: # if zero games in that season, skip season
      continue
    first_game = season_games.head(1).values[0]
    last_game = season_games.tail(1).values[0]

    if string_to_date(last_game) == rs_ends[i]:
      """
      option 1: player's season ends with player playing last game on the day the season ends
      """
      player[seasons[i]] = [string_to_date(first_game), string_to_date(last_game)]

    elif string_to_date(last_game) < rs_ends[i]:
      if player_injuries.shape[0] == 0:
        """
        option 2a: player's season ends when they play their last game before the season ends
        and they has never been injured so there cannot be a season ending injury
        """
        player[seasons[i]] = [string_to_date(first_game), string_to_date(last_game)]
      season_ending_injury = False
      for injury_date in player_injuries['date']:
        if (injury_date >= string_to_date(last_game)) and (injury_date <= rs_ends[i]):
          season_ending_injury = True
          """
          option 3: player's season ends with an injury between the last game they
          play in the season and the last day of the season (both days inclusive)
          """
          player[seasons[i]] = [string_to_date(first_game), injury_date]
      if not season_ending_injury:
        """
        option 2b: player's season ends when they play their last game before the season ends
        and they have been injured just not between their last game played this season
        and the official end day of the season
        """
        player[seasons[i]] = [string_to_date(first_game), string_to_date(last_game)]

  player_windows[pk] = player



100%|██████████| 1199/1199 [01:03<00:00, 18.80it/s]


In [ ]:
# applies the player_windows dictionary to filter the daily_fact.csv table

dfact['date'] = dfact['date'].apply(lambda x: string_to_date(x.split(" ")[0]))
prep = pd.DataFrame(columns=dfact.columns)

for pk in tqdm(dfact['player_key'].unique()):
  player_days = dfact.loc[dfact['player_key'] == pk, :]

  for season in list(player_windows[pk].keys()):
    start = player_windows[pk][season][0]
    end = player_windows[pk][season][1]

    season_days = player_days.loc[player_days['date'] >= start, :]
    season_days = season_days.loc[season_days['date'] <= end, :]

    prep = pd.concat([prep, season_days], axis=0, ignore_index=True)

print('Shape of dfact: ', dfact.shape)
print('Shape of prep: ', prep.shape)


  0%|          | 0/1199 [00:00<?, ?it/s]/tmp/ipython-input-2683343197.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prep = pd.concat([prep, season_days], axis=0, ignore_index=True)
100%|██████████| 1199/1199 [01:25<00:00, 13.95it/s]

Shape of dfact:  (1519374, 7)
Shape of prep:  (606995, 7)


In [ ]:
prep.head()

,player_key,team_key,date,age,experience,injured?,recoverying?
0,1713,23,2017-10-18,14875,7050,False,N
1,1713,23,2017-10-19,14876,7051,False,N
2,1713,23,2017-10-20,14877,7052,False,N
3,1713,23,2017-10-21,14878,7053,False,N
4,1713,23,2017-10-22,14879,7054,False,N


5. Remove days from the table where a player is recovering from a an injury

In [ ]:
prep['recoverying?'].value_counts() #verify that the 'E' values in recoverying column were removed naturally

,count
recoverying?,
N,537910
Y,69085


In [ ]:
prep = prep.loc[prep['recoverying?'] == 'N', :]
prep.drop('recoverying?', axis=1, inplace=True)
print(prep.shape)

(537910, 6)


6. Join player information that is unchanging across games

In [ ]:
prep = pd.merge(prep, pdim[['player_key', 'height', 'weight', 'draft_number']], on='player_key', how='inner')
prep.head()

,player_key,team_key,date,age,experience,injured?,height,weight,draft_number
0,1713,23,2017-10-18,14875,7050,False,198.12,220.0,5
1,1713,23,2017-10-19,14876,7051,False,198.12,220.0,5
2,1713,23,2017-10-20,14877,7052,False,198.12,220.0,5
3,1713,23,2017-10-21,14878,7053,False,198.12,220.0,5
4,1713,23,2017-10-22,14879,7054,False,198.12,220.0,5


In [ ]:
prep.isnull().sum() #found that weight is null in 200 records

,0
player_key,0
team_key,0
date,0
age,0
experience,0
injured?,0
height,0
weight,208
draft_number,5516


In [ ]:
players = []
prep.reset_index(inplace=True)

for idx in prep[prep['weight'].isnull()].index:
    if prep.loc[idx, 'player_key'] not in players:
        players.append(prep.loc[idx, 'player_key'])
print(players)

[1642449, 1642377]


In [ ]:
pdim.loc[pdim['player_key'] == players[0], ['first_name', 'last_name']]

,first_name,last_name
879,Tolu,Smith


In [ ]:
pdim.loc[pdim['player_key'] == players[1], ['first_name', 'last_name']]

,first_name,last_name
1154,Jaylen,Wells


In [ ]:
# basketball reference lists Tolu Smith (pk: 1642449) at 254lbs
# basketball reference lists Jaylen Wells (pk: 1642377) at 206lbs

insert_weight = {1642377:206.0, 1642449:254.0}
for idx in prep[prep['weight'].isnull()].index:
    for key in list(insert_weight.keys()):
        if prep.loc[idx, 'player_key'] == key:
            prep.loc[idx, 'weight'] = insert_weight[key]

prep.isnull().sum() # all good
# for now, no change to 'draft_number' as some players just don't participate in the draft

,0
index,0
player_key,0
team_key,0
date,0
age,0
experience,0
injured?,0
height,0
weight,0
draft_number,5516


In [ ]:
prep.to_csv('p1cp2.csv', index=True) #save preprocessing1 checkpoint 2, available in Preprocessing folder

Checkpoint 2

7. Join player information that is changing very slowly

In [ ]:
prep = pd.read_csv('/content/p1cp2.csv')
print('prep DataFrame shape before: ', prep.shape)

# lists for the loop to iterate through to retrieve information
player_keys = list(gfact['player_key'])
game_dates = list(gfact['game_date'])
team_keys = list(gfact['team_key'])
starters = list(gfact['starter'])
positions = list(gfact['player_position'])

# lists of arguments for the loop
prep_pks = list(prep['player_key'])
prep_dates = list(prep['date'])

# lists for loading information that belongs in the preprocessed table
prep_team_keys = []
prep_starter = []
prep_position = []

def get_recent_data(player_key, date):
  #get the date of the most recent game played in the season by a player and assign it to recent_game_date
  player_indices = []
  players_game_dates = []

  for i in range(len(game_dates)):
    if player_keys[i] == player_key:
      player_indices.append(i)
      players_game_dates.append(game_dates[i])

  if date in players_game_dates: #argument date is a day where an NBA game was played
    recent_game_date = date

  else:
    current_date = date
    while current_date not in players_game_dates:
      current_date = string_to_date(current_date) - timedelta(days=1)
      current_date = date_to_string(current_date)
    recent_game_date = current_date

  for i in player_indices:
    if game_dates[i] == recent_game_date:
      return [team_keys[i], starters[i], positions[i]]

for i in tqdm(range(len(prep_pks))):
  team_key, starter, position = get_recent_data(prep_pks[i], prep_dates[i])
  prep_team_keys.append(team_key)
  prep_starter.append(starter)
  prep_position.append(position)

temp = pd.DataFrame({
    'recent_team_key':prep_team_keys,
    'recent_starter':prep_starter,
    'recent_position':prep_position
    })

prep.reset_index(inplace=True)
prep = pd.concat([prep, temp], axis=1, ignore_index=True)
print('prep DataFrame shape after: ', prep.shape)
prep.head()


prep DataFrame shape before:  (537910, 11)


100%|██████████| 537910/537910 [1:52:32<00:00, 79.66it/s]


prep DataFrame shape after:  (537910, 15)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0,0,0,1713,23,2017-10-18,14875,7050,False,198.12,220.0,5,23,False,G
1,1,1,1,1713,23,2017-10-19,14876,7051,False,198.12,220.0,5,23,False,G
2,2,2,2,1713,23,2017-10-20,14877,7052,False,198.12,220.0,5,23,False,G
3,3,3,3,1713,23,2017-10-21,14878,7053,False,198.12,220.0,5,23,False,G
4,4,4,4,1713,23,2017-10-22,14879,7054,False,198.12,220.0,5,23,False,G


In [ ]:
prep.drop([0, 1, 2, 4], axis=1, inplace=True)
prep = prep.rename(columns={
    3:'player_key', 5:'date', 6: 'age', 7: 'experience',
    8: 'injured?', 9: 'height', 10: 'weight', 11: 'draft_number',
    12: 'recent_team_key', 13: 'recent_starter', 14: 'recent_position'})
prep.to_csv('p1cp3.csv', index=False) # preprocessing1 checkpoint 3 available in Preprocessing folder

Checkpoint 3

8. Join player information represented in a window of time

In [ ]:
prep = pd.read_csv('/content/p1cp3.csv')

In [ ]:
prep.head()

,player_key,date,age,experience,injured?,height,weight,draft_number,recent_team_key,recent_starter,recent_position
0,1713,2017-10-18,14875,7050,False,198.12,220.0,5,23,False,G
1,1713,2017-10-19,14876,7051,False,198.12,220.0,5,23,False,G
2,1713,2017-10-20,14877,7052,False,198.12,220.0,5,23,False,G
3,1713,2017-10-21,14878,7053,False,198.12,220.0,5,23,False,G
4,1713,2017-10-22,14879,7054,False,198.12,220.0,5,23,False,G


In [ ]:
# Start by adding history of injury status across past 16 weeks - only variable from daily_fact.csv

dfact['date'] = dfact['date'].apply(string_to_date) # allows for datetime math on dates

def get_injury_history(player_key, date, window_length):
  date = string_to_date(date)
  window_end = date - timedelta(days=window_length)

  temp = dfact.loc[dfact['player_key'] == player_key, ['date', 'injured?']]
  temp = temp.loc[temp['date'] > window_end, ['date', 'injured?']]
  temp = temp.loc[temp['date'] <= date, ['date', 'injured?']]

  oldest_date = pd.Timestamp(temp.head(1)['date'].values[0]) #date from top row - oldest record
  day_difference = oldest_date - window_end

  if day_difference.days > 1: #if there are days between the oldest record and oldest date in window
    injury_history = []
    for day in range(day_difference.days - 1):
      injury_history.append(temp.head(1)['injured?'].values[0]) #assume all older records are the same as the most recent
    for val in temp['injured?']:
      injury_history.append(val) # then add the recorded injury data
  else:
    injury_history = list(temp['injured?'])

  for i in range(len(injury_history)):
    if injury_history[i] == np.True_:
      injury_history[i] = True
    if injury_history[i] == np.False_:
      injury_history[i] = False

  return injury_history

injury_histories = []
for pk in tqdm(prep['player_key'].unique()):
  player_dates = prep.loc[prep['player_key'] == pk, 'date']
  for date in player_dates:
    injury_history = get_injury_history(pk, date, 112)
    injury_histories.append(injury_history)

100%|██████████| 1195/1195 [2:12:55<00:00,  6.67s/it]


In [ ]:
column_names = []
for i in range(112):
  column_names.append("-" + str(i) + "_days_injury_status")
column_names.reverse()

injury_history_df = pd.DataFrame(data=injury_histories, columns=column_names)

In [ ]:
prep = pd.concat([prep, injury_history_df], axis=1)
print(prep.shape)
print(prep.isnull().sum())

(537910, 123)
player_key               0
date                     0
age                      0
experience               0
injured?                 0
                        ..
-4_days_injury_status    0
-3_days_injury_status    0
-2_days_injury_status    0
-1_days_injury_status    0
-0_days_injury_status    0
Length: 123, dtype: int64


In [ ]:
prep.to_csv('p1cp4.csv', index=False) # preprocessing1 checkpoint 4 available in Preprocessing folder

In [ ]:
prep.shape

(537910, 123)

Checkpoint 4

In [ ]:
prep = pd.read_csv('/content/p1cp4.csv')

/tmp/ipython-input-1776454485.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  prep = pd.read_csv('/content/p1cp4.csv')


In [ ]:
# next, get the 2 week window variables from the game_fact.csv table

all_new_rows = []

for pk in tqdm(prep['player_key'].unique()):
  player_dates = prep.loc[prep['player_key'] == pk, 'date']

  games = gfact.loc[gfact['player_key'] == pk, ['game_date', 'opposing_team_key', 'home?', 'location_arena_key', 'arena_lat', 'arena_long']]
  for date in player_dates:
    window_end_date = string_to_date(date) - timedelta(days=14)
    opposing_team_keys = []
    home_status = []
    arena_keys = []
    arena_lats = []
    arena_longs = []

    while string_to_date(date) > window_end_date:
      if date in games['game_date'].values: # if game was played on that date,
        opposing_team_key = games.loc[games['game_date'] == date, 'opposing_team_key'].values[0]
        opposing_team_keys.append(opposing_team_key)

        home = games.loc[games['game_date'] == date, 'home?'].values[0]
        home_status.append(home)

        arena_key = games.loc[games['game_date'] == date, 'location_arena_key'].values[0]
        arena_keys.append(arena_key)

        arena_lat = games.loc[games['game_date'] == date, 'arena_lat'].values[0]
        arena_lats.append(arena_lat)

        arena_long = games.loc[games['game_date'] == date, 'arena_long'].values[0]
        arena_longs.append(arena_long)

      else: #if game wasn't played on that date, add default values
        opposing_team_keys.append(-1)
        home_status.append(-1)
        arena_keys.append(-1)
        arena_lats.append(-1)
        arena_longs.append(-1)
      new_date = string_to_date(date) - timedelta(days=1)
      date = date_to_string(new_date)

    new_row = opposing_team_keys + home_status + arena_keys + arena_lats + arena_longs
    all_new_rows.append(new_row)


100%|██████████| 1195/1195 [1:29:54<00:00,  4.51s/it]


In [ ]:
columns = []
for i in range(14):
  columns.append("-" + str(i) + "_days_opposing_team_key")
for i in range(14):
  columns.append("-" + str(i) + "_days_home?")
for i in range(14):
  columns.append("-" + str(i) + "_days_arena_key")
for i in range(14):
  columns.append("-" + str(i) + "_days_arena_lat")
for i in range(14):
  columns.append("-" + str(i) + "_days_arena_long")
print(columns)

['-0_days_opposing_team_key', '-1_days_opposing_team_key', '-2_days_opposing_team_key', '-3_days_opposing_team_key', '-4_days_opposing_team_key', '-5_days_opposing_team_key', '-6_days_opposing_team_key', '-7_days_opposing_team_key', '-8_days_opposing_team_key', '-9_days_opposing_team_key', '-10_days_opposing_team_key', '-11_days_opposing_team_key', '-12_days_opposing_team_key', '-13_days_opposing_team_key', '-0_days_home?', '-1_days_home?', '-2_days_home?', '-3_days_home?', '-4_days_home?', '-5_days_home?', '-6_days_home?', '-7_days_home?', '-8_days_home?', '-9_days_home?', '-10_days_home?', '-11_days_home?', '-12_days_home?', '-13_days_home?', '-0_days_arena_key', '-1_days_arena_key', '-2_days_arena_key', '-3_days_arena_key', '-4_days_arena_key', '-5_days_arena_key', '-6_days_arena_key', '-7_days_arena_key', '-8_days_arena_key', '-9_days_arena_key', '-10_days_arena_key', '-11_days_arena_key', '-12_days_arena_key', '-13_days_arena_key', '-0_days_arena_lat', '-1_days_arena_lat', '-2_day

In [ ]:
temp = pd.DataFrame(data=all_new_rows, columns=columns)
prep = pd.concat([prep, temp], axis=1)
prep.head()

,player_key,date,age,experience,injured?,height,weight,draft_number,recent_team_key,recent_starter,...,-4_days_arena_long,-5_days_arena_long,-6_days_arena_long,-7_days_arena_long,-8_days_arena_long,-9_days_arena_long,-10_days_arena_long,-11_days_arena_long,-12_days_arena_long,-13_days_arena_long
0,1713,2017-10-18,14875,7050,False,198.12,220.0,5,23,False,...,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
1,1713,2017-10-19,14876,7051,False,198.12,220.0,5,23,False,...,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2,1713,2017-10-20,14877,7052,False,198.12,220.0,5,23,False,...,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
3,1713,2017-10-21,14878,7053,False,198.12,220.0,5,23,False,...,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
4,1713,2017-10-22,14879,7054,False,198.12,220.0,5,23,False,...,121.499444,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0


In [ ]:
prep.to_csv('p1cp5.csv', index=False) # preprocessing1 checkpoint 5 available in Preprocessing folder

# this file will be one of the exploratory data analysis files: eda_data1

Checkpoint 5

In [ ]:
prep = pd.read_csv('/content/p1cp5.csv')

/tmp/ipython-input-3385434587.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  prep = pd.read_csv('/content/p1cp5.csv')


In [ ]:
def index_item(ls, key):
  for i in range(len(ls)):
    if ls[i] == key:
      return i
  return -1

#alternate strategy: rewrite the code but with lists

all_new_rows = []

for pk in tqdm(prep['player_key'].unique()):
  player_dates = prep.loc[prep['player_key'] == pk, 'date']

  relevant_cols = [
      'game_date', 'minutes', 'did_not_play',
      'distance_traveled', 'field_goals_attempted',
      'three_point_field_goals_attempted',
      'free_throws_attempted', 'defensive_rebounds',
      'offensive_rebounds', 'rebounds', 'blocks',
      'fouls']

  game_date_ls = list(gfact.loc[gfact['player_key'] == pk, 'game_date'])
  minutes_ls = list(gfact.loc[gfact['player_key'] == pk, 'minutes'])
  did_not_play_ls = list(gfact.loc[gfact['player_key'] == pk, 'did_not_play'])
  distance_traveled_ls = list(gfact.loc[gfact['player_key'] == pk, 'distance_traveled'])
  fga_ls = list(gfact.loc[gfact['player_key'] == pk, 'field_goals_attempted'])
  fga3_ls = list(gfact.loc[gfact['player_key'] == pk, 'three_point_field_goals_attempted'])
  defensive_rebounds_ls = list(gfact.loc[gfact['player_key'] == pk, 'defensive_rebounds'])
  offensive_rebounds_ls = list(gfact.loc[gfact['player_key'] == pk, 'offensive_rebounds'])
  rebounds_ls = list(gfact.loc[gfact['player_key'] == pk, 'rebounds'])
  blocks_ls = list(gfact.loc[gfact['player_key'] == pk, 'blocks'])
  fouls_ls = list(gfact.loc[gfact['player_key'] == pk, 'fouls'])

  for date in player_dates:
    window_end_date = string_to_date(date) - timedelta(days=56)
    minutes_history = []
    did_not_play_history = []
    distance_traveled_history = []
    fga_history = []
    three_points_fga_history = []
    def_rebounds_history = []
    off_rebounds_history = []
    rebounds_history = []
    blocks_history = []
    fouls_history = []

    while string_to_date(date) > window_end_date:

      key = index_item(game_date_ls, date)

      if key != -1: # if game was played on that date, get values and append them to matching lists

        minutes = minutes_ls[key] #get value
        if np.isnan(minutes): #if the value is nan, replace with zero
          minutes = np.float64(0)
        minutes_history.append(minutes) #add the value to the matching list

        did_not_play = did_not_play_ls[key]
        if np.isnan(did_not_play):
          did_not_play = np.float64(0)
        did_not_play_history.append(did_not_play)

        distance_traveled = distance_traveled_ls[key]
        if np.isnan(distance_traveled):
          distance_traveled = np.float64(0)
        distance_traveled_history.append(distance_traveled)

        fga = fga_ls[key]
        if np.isnan(fga):
          fga = np.float64(0)
        fga_history.append(fga)

        three_point_fga = fga3_ls[key]
        if np.isnan(three_point_fga):
          three_point_fga = np.float64(0)
        three_points_fga_history.append(three_point_fga)

        def_rebounds = defensive_rebounds_ls[key]
        if np.isnan(def_rebounds):
          def_rebounds = np.float64(0)
        def_rebounds_history.append(def_rebounds)

        off_rebounds = offensive_rebounds_ls[key]
        if np.isnan(off_rebounds):
          off_rebounds = np.float64(0)
        off_rebounds_history.append(off_rebounds)

        rebounds = rebounds_ls[key]
        if np.isnan(rebounds):
          rebounds = np.float64(0)
        rebounds_history.append(rebounds)

        blocks = blocks_ls[key]
        if np.isnan(blocks):
          blocks = np.float64(0)
        blocks_history.append(blocks)

        fouls = fouls_ls[key]
        if np.isnan(fouls):
          fouls = np.float64(0)
        fouls_history.append(fouls)

      else: #if game wasn't played on that date, add default values
        minutes_history.append(-1)
        did_not_play_history.append(-1)
        distance_traveled_history.append(-1)
        fga_history.append(-1)
        three_points_fga_history.append(-1)
        def_rebounds_history.append(-1)
        off_rebounds_history.append(-1)
        rebounds_history.append(-1)
        blocks_history.append(-1)
        fouls_history.append(-1)

      new_date = string_to_date(date) - timedelta(days=1)
      date = date_to_string(new_date)

    new_row = minutes_history + did_not_play_history + distance_traveled_history + fga_history + three_points_fga_history + def_rebounds_history + off_rebounds_history + rebounds_history + blocks_history + fouls_history
    all_new_rows.append(new_row)

100%|██████████| 1195/1195 [21:26<00:00,  1.08s/it]


In [ ]:
# notes about data outcome:
"""
- distance traveled is ignored when the player is injured, not accurate
- zeros in the data are np.float() not just regular float
"""

In [ ]:
col_names = []
for i in range(56):
    col_names.append("-" + str(i) + '_days_minutes')
for i in range(56):
    col_names.append("-" + str(i) + '_days_did_not_play')
for i in range(56):
    col_names.append("-" + str(i) + '_days_distance_traveled')
for i in range(56):
    col_names.append("-" + str(i) + '_days_field_goals_attempted')
for i in range(56):
    col_names.append("-" + str(i) + '_days_3pt_field_goals_attempted')
for i in range(56):
    col_names.append("-" + str(i) + '_days_def_rebounds')
for i in range(56):
    col_names.append("-" + str(i) + '_days_off_rebounds')
for i in range(56):
    col_names.append("-" + str(i) + '_days_rebounds')
for i in range(56):
    col_names.append("-" + str(i) + '_days_blocks')
for i in range(56):
    col_names.append("-" + str(i) + '_days_fouls')
print(len(col_names))

In [ ]:
o1_data = [row[:112] for row in all_new_rows]
o1 = pd.DataFrame(data=o1_data, columns=col_names[:112])
o1 = pd.concat([o1, prep[['injured?']]], axis=1)
o1.to_csv('C:\\Users\\oscar\\big_ahh_files\\o1.csv', index=False)
# this file will be one of the exploratory data analysis files: eda_data2

In [ ]:
o2_data = [row[112:224] for row in all_new_rows]
o2 = pd.DataFrame(data=o2_data, columns=col_names[112:224])
o2 = pd.concat([o2, prep[['injured?']]], axis=1)
o2.to_csv('C:\\Users\\oscar\\big_ahh_files\\o2.csv', index=False)
# this file will be one of the exploratory data analysis files: eda_data3

In [ ]:
o3_data = [row[224:336] for row in all_new_rows]
o3 = pd.DataFrame(data=o3_data, columns=col_names[224:336])
o3 = pd.concat([o3, prep[['injured?']]], axis=1)
o3.to_csv('C:\\Users\\oscar\\big_ahh_files\\o3.csv', index=False)
# this file will be one of the exploratory data analysis files: eda_data4

In [ ]:
o4_data = [row[336:448] for row in all_new_rows]
o4 = pd.DataFrame(data=o4_data, columns=col_names[336:448])
o4 = pd.concat([o4, prep[['injured?']]], axis=1)
o4.to_csv('C:\\Users\\oscar\\big_ahh_files\\o4.csv', index=False)
# this file will be one of the exploratory data analysis files: eda_data5

In [ ]:
o5_data = [row[448:560] for row in all_new_rows]
o5 = pd.DataFrame(data=o5_data, columns=col_names[448:560])
o5 = pd.concat([o5, prep[['injured?']]], axis=1)
o5.to_csv('C:\\Users\\oscar\\big_ahh_files\\o5.csv', index=False)
# this file will be one of the exploratory data analysis files: eda_data6